# 02 — Parsing and Segments

This notebook covers the structured-data pipeline that transforms raw
SEC data into analysis-ready tables:

1. **XBRL parsing** — companyfacts JSON → structured `FinancialFact` rows
2. **Concept resolution** — priority-ordered fallback across XBRL tag names
3. **Deduplication** — prefer 10-K over 10-Q, latest filing over amendments
4. **Validation** — compare parsed values against published 10-K figures
5. **Data Validation Gate** — hard gate that blocks formal recommendations on bad data
6. **Segment normalization** — map Nvidia's changing labels to consistent categories
7. **Filing text extraction** — Item-number regex with title fallback
8. **Data quality reporting** — missing tags, fallbacks, coverage

Modules: `src/xbrl_parser.py`, `src/data_validation.py`, `src/segment_revenue.py`, `src/filing_text_parser.py`

---

### ⚠️ Data Validation Gate (Post-Mortem Addition)

After XBRL parsing, the pipeline runs a **data validation gate** that cross-checks
every core metric against published 10-K values (1% tolerance). This was added after
the initial pipeline produced a catastrophically wrong Sell recommendation because
FY2025 revenue was parsed as $27B instead of $130B (a quarterly value was selected
instead of the annual total).

The gate produces a `DataQualityStatus`:
- **PASS** — all core metrics within tolerance → formal rating eligible
- **PASS_WITH_WARNINGS** — non-critical deviations → formal rating eligible with disclosures
- **DATA_BLOCKED** — critical metric failure → report becomes 'Not Rated — Data Validation Required'

Key components:
- `src/data_validation.py` — `DataValidationGate` class
- `src/xbrl_parser.py` — `validate_against_published()`, `compute_data_quality_status()`
- Validated metrics: revenue, gross_profit, operating_income, net_income, operating_cash_flow,
  capex, FCF, cash_and_securities, total_debt, diluted_shares, diluted_EPS, R&D
- Published reference values: `tests/fixtures/known_validation_values.json`
- Output: `outputs/data_quality_report.md`, `outputs/audit_status.json`

> **Note:** The authoritative reproducibility path is `python scripts/run_pipeline.py --ticker NVDA --report-date 2026-05-01 --price-date 2026-05-01 --output-format both`. This notebook is an explanatory wrapper that inspects the same pipeline outputs.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import json
from pathlib import Path

from src.config import get_default_config
from src.xbrl_parser import XBRLParser
from src.segment_revenue import SegmentRevenueNormalizer
from src.filing_text_parser import FilingTextParser

In [ ]:
config = get_default_config()

## 1. XBRL Parsing

The `XBRLParser` reads the companyfacts JSON and extracts every required
line item (revenue, COGS, net income, FCF components, etc.) using a
priority-ordered `CONCEPT_MAP`. If the primary XBRL tag is missing,
it falls back to alternatives — and logs the fallback.

In [ ]:
parser = XBRLParser(config)

# Show the concept map — each metric has prioritised XBRL tag names
print("Metrics tracked:", list(parser.CONCEPT_MAP.keys()))
print(f"\nExample — 'revenue' concepts (in priority order):")
for i, concept in enumerate(parser.CONCEPT_MAP["revenue"], 1):
    print(f"  {i}. {concept}")

In [ ]:
# Load cached companyfacts (requires prior ingestion run)
facts_path = config.raw_dir / f"companyfacts_CIK{config.cik.lstrip('0').zfill(10)}.json"

if facts_path.exists():
    with open(facts_path) as f:
        facts_json = json.load(f)
    parsed = parser.parse_companyfacts(facts_json)
    print(f"Parsed {len(parsed)} financial facts")
    print(f"Metrics found: {parsed['metric_name'].nunique()}")
    print(f"Fiscal years: {sorted(parsed['fiscal_year'].unique())}")
    parsed.head(10)
else:
    print("No cached companyfacts — run 01_data_ingestion first.")
    parsed = None

## 2. Validation Against Published Values

We compare parsed XBRL values against known published 10-K figures
(stored in `tests/fixtures/known_validation_values.json`). A tolerance
threshold determines pass/fail/missing status.

In [ ]:
known_path = config.fixtures_dir / "known_validation_values.json"

if parsed is not None and known_path.exists():
    with open(known_path) as f:
        known_values = json.load(f)
    validation = parser.validate_against_published(parsed, known_values)
    print(f"Validation results: {len(validation)} checks")
    print(f"  Pass:    {(validation['status'] == 'pass').sum()}")
    print(f"  Fail:    {(validation['status'] == 'fail').sum()}")
    print(f"  Missing: {(validation['status'] == 'missing').sum()}")
    validation
else:
    print("Skipping validation — parsed data or known values not available.")

## 3. Data Validation Gate

The `DataValidationGate` (in `src/data_validation.py`) is the hard gate
between XBRL parsing and all downstream pipeline stages. It determines
whether the pipeline can issue a formal Buy/Hold/Sell recommendation.

**How it works:**
1. For each core metric × fiscal year, compare `parsed_value` against `published_value`
2. Compute `diff_pct = abs((parsed - published) / published) × 100`
3. If `diff_pct > tolerance` (default 1%) for a critical metric → `DATA_BLOCKED`
4. Also checks fiscal-year selection: flags any fact where `duration_days < 340` was used as annual

The gate also verifies `validation_coverage_pct` — if fewer than 90% of core
metric/year pairs are validated for the latest 3 fiscal years, the status is `DATA_BLOCKED`.

**Downstream impact of DATA_BLOCKED:**
- All valuation outputs labeled "Diagnostic only"
- Report cover page shows "Not Rated — Data Validation Required"
- `audit_status.json` records the blocking issues
- Pipeline continues (for diagnostic value) but no formal rating is issued

In [ ]:
from src.data_validation import DataValidationGate

if parsed is not None and known_path.exists():
    gate = DataValidationGate(config)
    with open(known_path) as f:
        known_values = json.load(f)
    status, validated_metrics = gate.validate_parsed_data(parsed, known_values)
    print(f"DataQualityStatus: {status}")
    print(f"Should block recommendation: {gate.should_block_recommendation(status)}")
    print(f"Diagnostic label: {gate.get_diagnostic_label(status)}")
    
    # Check fiscal-year selection correctness
    fy_issues = gate.check_fiscal_year_selection(parsed)
    if fy_issues:
        print(f"\nFiscal-year selection issues: {len(fy_issues)}")
        for issue in fy_issues[:5]:
            print(f"  [{issue.severity}] {issue.detail}")
    else:
        print("\nNo fiscal-year selection issues found.")
else:
    print("Skipping validation gate — parsed data or known values not available.")

## 4. Data Quality Report

The parser tracks missing tags, fallback tags used, and coverage.
This feeds into `outputs/data_quality_report.md`.

In [ ]:
if parsed is not None:
    validation_df = validation if 'validation' in dir() else __import__('pandas').DataFrame()
    report_md = parser.generate_data_quality_report(parsed, validation_df)
    # Show first 40 lines
    for line in report_md.split("\n")[:40]:
        print(line)

## 5. Segment Revenue Normalization

Nvidia changed its reportable segments and market/platform categories
multiple times between FY2016 and FY2026. The `SegmentRevenueNormalizer`
maps original labels to consistent categories using `LABEL_MAPPING`.

Primary extraction uses XBRL dimensions; table extraction from filing
text is the fallback.

In [ ]:
normalizer = SegmentRevenueNormalizer(config)

# Show the label mapping
print("Label mapping (original → normalized):")
for orig, norm in sorted(normalizer.LABEL_MAPPING.items()):
    print(f"  {orig:40s} → {norm}")

In [ ]:
import pandas as pd

# Load pre-computed segment data if available
seg_path = config.processed_dir / "nvda_segment_revenue_normalized.csv"
if seg_path.exists():
    segments = pd.read_csv(seg_path)
    print(f"Segment records: {len(segments)}")
    print(f"Categories: {segments['normalized_category'].unique().tolist()}")
    print(f"Extraction methods: {segments['extraction_method'].unique().tolist()}")
    segments.head(10)
else:
    print("No segment data yet — run the pipeline first.")

## 6. Filing Text Extraction

The `FilingTextParser` extracts narrative sections from filing HTML:
- **10-K**: Item 1 (Business), Item 1A (Risk Factors), Item 7 (MD&A), Item 7A
- **10-Q**: Part II Item 1A (Risk Factors), Part I Item 2 (MD&A)

Primary method: Item-number regex. Fallback: section-title matching.
Each section is stored as a `TextSectionRecord` with `source_available_date`.

In [ ]:
text_parser = FilingTextParser(config)

print("10-K sections:", text_parser.SECTIONS_10K)
print("10-Q sections:", text_parser.SECTIONS_10Q)

In [ ]:
# Parse a fixture 10-K snippet as a demonstration
fixture_10k = config.fixtures_dir / "sample_10k_snippet.html"
if fixture_10k.exists():
    html = fixture_10k.read_text(encoding="utf-8")
    sections = text_parser.parse_filing(
        html=html,
        accession="test_accession",
        form_type="10-K",
        filing_date="2025-02-26",
        source_available_date="2025-02-26",
    )
    for sec in sections:
        print(f"  {sec.section_name:20s}  chars={sec.char_count:>6,}  status={sec.parse_status}")
else:
    print("No 10-K fixture available.")

In [ ]:
# Generate coverage report
if 'sections' in dir() and sections:
    coverage = text_parser.generate_coverage_report(sections)
    coverage

---

**Next:** [03_financial_analysis_and_nlp.ipynb](03_financial_analysis_and_nlp.ipynb) —
financial metrics computation and narrative drift analysis.